In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
# uncomment to use hosted db
del os.environ["BIRDDOG_USE_LOCAL_NOCODB"]
os.environ.get("BIRDDOG_USE_LOCAL_NOCODB")

In [3]:
from birddog.database import Database
from birddog.wiki import (
    page_label,
    sequential_page_label,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
)

2026-04-24 07:45:43,183 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-04-24 07:45:43,325 [INFO] Translation is enabled. Using GCP translator
2026-04-24 07:45:43,326 [INFO] Using Google Cloud translation API
2026-04-24 07:45:43,326 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-04-24 07:45:44,146 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     3.55    39.00       0.00           24


In [ ]:
pages = db.get_all_ids("Pages")

In [5]:
r,_ = db.scan("Documents", view_name="BD:WDT:commons.wikimedia.org")

In [6]:
db._view_id_map

{'Documents': {'All Documents': 'vwxo82wt4e2rmvpz',
  'Untranslated Documents': 'vwh4lfvqoe2jt3b5',
  'Recently Imported Records': 'vw09clo4piapqbme',
  'Recently Updated Records': 'vw8fj4kh6lza7s2d',
  'Birddog Updates': 'vw28yd7ycepvpopr',
  'DAOO-D': 'vwzr5lf1c8jjijjh',
  'DAOO-R': 'vwsz2egnhmfit6ov',
  'WikiDocTracker: uk.wikisource.org': 'vwmqoo63x1gz51wg',
  'WikiDocTracker: commons.wikimedia.org': 'vwm88gn9a3qksr8n',
  'BD:Untranslated': 'vwmbwpc2i74fwj52',
  'BD:WDT:uk.wikisource.org': 'vwwomhqdru3l4em9',
  'BD:WDT:commons.wikimedia.org': 'vw04nzz11z9qvudk'}}

In [ ]:
len(pages)

In [ ]:
db.delete("Pages", pages)

In [ ]:
docs = db.get_all_ids("Documents")

In [ ]:
len(docs)

In [ ]:
db.delete("Documents", docs)

In [ ]:
pages = []
cursor = None
while True:
    if cursor and (int(cursor) % 10000) == 0:
        print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=1000,
        where=("seq_label", "is", None),
        fields=("Id","title","label","seq_label"),
    )
    if batch:
        pages.extend(batch)
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
pages[1000]

In [ ]:
def normalize_labels(page):
    result = page.copy()
    title = page.get("title")
    if title:
        proper_label = page_label(title)
        result["label"] = proper_label
        proper_seq_label = sequential_page_label(proper_label)
        result["seq_label"] = proper_seq_label
    return result

In [ ]:
normalize_labels(pages[2000])

In [ ]:
pages[2000]

In [ ]:
pages[2000] == normalize_labels(pages[2000])

In [ ]:
pages[2000] == pages[2000].copy()

In [ ]:
norm_pages = [normalize_labels(p) for p in pages]

In [ ]:
norm_pages[:10]

In [ ]:
changed_pages = [n for n,p in zip(norm_pages, pages) if n != p]

In [ ]:
len(changed_pages)

In [ ]:
len(pages)

In [ ]:
len(norm_pages)

In [ ]:
rec_ids = db.write("Pages", norm_pages[1000:2000])

In [ ]:
chunk = 1000
for i in range(0, len(norm_pages), chunk):
    print(i)
    rec_ids = db.write("Pages", norm_pages[i:(i+chunk)])